In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Define Paths and Table Names
raw_path = "abfss://raw@stlinggarprojectdev001.dfs.core.windows.net/nyc_ridehailing/nyc_trips_latest.json"

# THIS is your physical storage location
bronze_location = "abfss://bronze@stlinggarprojectdev001.dfs.core.windows.net/"

# THIS is your Unity Catalog table name (No slashes, no abfss://)
bronze_table_name = "dev_bronze.default.bronze_ridehailing"

# 2. Read JSON Lines
df_raw = (spark.read
    .format("json")
    .load(raw_path)
    .select("*", "_metadata.file_path") 
    .withColumnRenamed("file_path", "source_file") 
    .withColumn("ingestion_time", current_timestamp()))

# 3. Write to Bronze
# We use 'path' to tell Spark where to put the files, 
# and 'saveAsTable' to give it a clean name in the Catalog.
(df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", None) # Physical storage
    .saveAsTable(bronze_table_name, mode='overwrite') # Catalog registration
)
print(f"Step 1 Complete: {spark.read.table(bronze_table_name).count()} rows moved to Bronze.")